<a href="https://colab.research.google.com/github/nicholas-schwier-class/Stats-Sum-26/blob/main/Class_8_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Importing Libraries and Creating Sample Data

In [21]:
import pandas as pd
import numpy as np

# Reproducible results
np.random.seed(42)

df = pd.DataFrame({
    "date": pd.date_range("2025-01-01", periods=100),
    "product": np.random.choice(
        ["Laptop", "Tablet", "Phone"], 100
    ),
    "region": np.random.choice(
        ["East", "West", "North"], 100
    ),
    "sales": np.random.randint(100, 5000, 100)
})

print(df.head())

        date product region  sales
0 2025-01-01   Phone  North   3863
1 2025-01-02  Laptop  North   1953
2 2025-01-03   Phone  North   4759
3 2025-01-04   Phone   East   1391
4 2025-01-05  Laptop  North   3681


Multi-Level Indexing (MultiIndex)

Why use MultiIndex?

A MultiIndex allows multiple columns to become the index, making hierarchical data easier to analyze.

In [22]:
multi_df = df.set_index(["region", "product"])

print(multi_df.head())

                     date  sales
region product                  
North  Phone   2025-01-01   3863
       Laptop  2025-01-02   1953
       Phone   2025-01-03   4759
East   Phone   2025-01-04   1391
North  Laptop  2025-01-05   3681


Selecting Data from a MultiIndex

In [23]:
east_laptops = multi_df.loc[("East", "Laptop")]

print(east_laptops.head())

                     date  sales
region product                  
East   Laptop  2025-01-13   4197
       Laptop  2025-01-21    302
       Laptop  2025-01-26   4489
       Laptop  2025-02-14   3011
       Laptop  2025-02-21   3019


/tmp/ipykernel_2406/343800457.py:1: PerformanceWarning: indexing past lexsort depth may impact performance.
  east_laptops = multi_df.loc[("East", "Laptop")]


Advanced Filtering
Traditional Boolean Filtering

In [24]:
filtered = df[
    (df["sales"] > 3000) &
    (df["region"] == "East")
]

print(filtered.head())

         date product region  sales
8  2025-01-09   Phone   East   3099
12 2025-01-13  Laptop   East   4197
25 2025-01-26  Laptop   East   4489
27 2025-01-28   Phone   East   3031
32 2025-02-02  Tablet   East   3372


Using Query Syntax

Many analysts find query() easier to read.

In [25]:
filtered = df.query(
    "sales > 3000 and region == 'East'"
)

print(filtered.head())

         date product region  sales
8  2025-01-09   Phone   East   3099
12 2025-01-13  Laptop   East   4197
25 2025-01-26  Laptop   East   4489
27 2025-01-28   Phone   East   3031
32 2025-02-02  Tablet   East   3372


GroupBy Operations
Average Sales per Product

In [26]:
grouped = (
    df.groupby("product")["sales"]
      .mean()
)

print(grouped)

product
Laptop    2617.212121
Phone     2384.870968
Tablet    3087.444444
Name: sales, dtype: float64


In [27]:
#MULTIPLE AGGREGATIONS
summary = (
    df.groupby("product")
      .agg({
          "sales": [
              "mean",
              "max",
              "min",
              "std"
          ]
      })
)

print(summary)

               sales                        
                mean   max  min          std
product                                     
Laptop   2617.212121  4596  198  1409.747445
Phone    2384.870968  4837  246  1506.698128
Tablet   3087.444444  4836  954  1101.444336


Transform vs Aggregate

Aggregate Reduces Rows

In [28]:
df.groupby("product")["sales"].mean()

,sales
product,
Laptop,2617.212121
Phone,2384.870968
Tablet,3087.444444


In [29]:
#TRANSFORM PRESERVES ROWS
df["product_avg"] = (
    df.groupby("product")["sales"]
      .transform("mean")
)

print(df.head())

        date product region  sales  product_avg
0 2025-01-01   Phone  North   3863  2384.870968
1 2025-01-02  Laptop  North   1953  2617.212121
2 2025-01-03   Phone  North   4759  2384.870968
3 2025-01-04   Phone   East   1391  2384.870968
4 2025-01-05  Laptop  North   3681  2617.212121


In [30]:
#Comparing Each Sale to Product Average
df["difference"] = (
    df["sales"] -
    df["product_avg"]
)

print(df.head())

        date product region  sales  product_avg   difference
0 2025-01-01   Phone  North   3863  2384.870968  1478.129032
1 2025-01-02  Laptop  North   1953  2617.212121  -664.212121
2 2025-01-03   Phone  North   4759  2384.870968  2374.129032
3 2025-01-04   Phone   East   1391  2384.870968  -993.870968
4 2025-01-05  Laptop  North   3681  2617.212121  1063.787879


Pivot Tables

Pivot tables summarize large datasets quickly.

In [31]:
pivot = pd.pivot_table(
    df,
    values="sales",
    index="region",
    columns="product",
    aggfunc="mean"
)

print(pivot)

product       Laptop        Phone       Tablet
region                                        
East     2808.454545  1890.400000  2799.000000
North    2722.125000  2748.857143  3131.333333
West     1986.833333  2363.285714  3332.000000


Merging DataFrames

Additional Dataset

In [32]:
products = pd.DataFrame({
    "product": ["Laptop", "Tablet", "Phone"],
    "category": ["Computers", "Mobile", "Mobile"]
})

In [33]:
#LEFT JOIN
merged = pd.merge(
    df,
    products,
    on="product",
    how="left"
)

print(merged.head())

        date product region  sales  product_avg   difference   category
0 2025-01-01   Phone  North   3863  2384.870968  1478.129032     Mobile
1 2025-01-02  Laptop  North   1953  2617.212121  -664.212121  Computers
2 2025-01-03   Phone  North   4759  2384.870968  2374.129032     Mobile
3 2025-01-04   Phone   East   1391  2384.870968  -993.870968     Mobile
4 2025-01-05  Laptop  North   3681  2617.212121  1063.787879  Computers


VERTICAL CONCATENATION

In [34]:
df1 = pd.DataFrame({"A": [1, 2]})
df2 = pd.DataFrame({"A": [3, 4]})

combined = pd.concat([df1, df2])

print(combined)

   A
0  1
1  2
0  3
1  4


Working with Time Series

---



In [35]:
#Convert Date Column into Index
time_df = df.set_index("date")

#Monthly Sales
monthly_sales = (
    time_df["sales"]
    .resample("ME")
    .sum()
)

print(monthly_sales)

#Rolling Average
time_df["rolling_7"] = (
    time_df["sales"]
    .rolling(window=7)
    .mean()
)

print(time_df.head(10))

date
2025-01-31    87227
2025-02-28    63085
2025-03-31    94140
2025-04-30    26995
Freq: ME, Name: sales, dtype: int64
           product region  sales  product_avg   difference    rolling_7
date                                                                   
2025-01-01   Phone  North   3863  2384.870968  1478.129032          NaN
2025-01-02  Laptop  North   1953  2617.212121  -664.212121          NaN
2025-01-03   Phone  North   4759  2384.870968  2374.129032          NaN
2025-01-04   Phone   East   1391  2384.870968  -993.870968          NaN
2025-01-05  Laptop  North   3681  2617.212121  1063.787879          NaN
2025-01-06  Laptop  North   3557  2617.212121   939.787879          NaN
2025-01-07   Phone   East   1736  2384.870968  -648.870968  2991.428571
2025-01-08  Tablet  North   3796  3087.444444   708.555556  2981.857143
2025-01-09   Phone   East   3099  2384.870968   714.129032  3145.571429
2025-01-10   Phone   West   3252  2384.870968   867.129032  2930.285714


Window Functions

In [36]:
#CUMULATIVE SUM
df["cumulative_sales"] = (
    df["sales"]
    .cumsum()
)

print(df.head())

        date product region  sales  product_avg   difference  cumulative_sales
0 2025-01-01   Phone  North   3863  2384.870968  1478.129032              3863
1 2025-01-02  Laptop  North   1953  2617.212121  -664.212121              5816
2 2025-01-03   Phone  North   4759  2384.870968  2374.129032             10575
3 2025-01-04   Phone   East   1391  2384.870968  -993.870968             11966
4 2025-01-05  Laptop  North   3681  2617.212121  1063.787879             15647


In [37]:
#Ranking Within Groups
df["sales_rank"] = (
    df.groupby("product")["sales"]
      .rank(ascending=False)
)

print(df.head())

        date product region  sales  product_avg   difference  \
0 2025-01-01   Phone  North   3863  2384.870968  1478.129032   
1 2025-01-02  Laptop  North   1953  2617.212121  -664.212121   
2 2025-01-03   Phone  North   4759  2384.870968  2374.129032   
3 2025-01-04   Phone   East   1391  2384.870968  -993.870968   
4 2025-01-05  Laptop  North   3681  2617.212121  1063.787879   

   cumulative_sales  sales_rank  
0              3863         7.0  
1              5816        21.0  
2             10575         2.0  
3             11966        23.0  
4             15647        10.0  


Handling Missing Data

In [38]:
missing_df = pd.DataFrame({
    "A": [1, np.nan, 3, np.nan],
    "B": [5, 6, np.nan, 8]
})

print(missing_df)

     A    B
0  1.0  5.0
1  NaN  6.0
2  3.0  NaN
3  NaN  8.0


In [39]:
#DETECT MISSING VALUES
print(
    missing_df.isna().sum()
)

A    2
B    1
dtype: int64


In [40]:
#Fill Missing Values
filled = missing_df.fillna(0)

print(filled)

     A    B
0  1.0  5.0
1  0.0  6.0
2  3.0  0.0
3  0.0  8.0


Applying Custom Functions

Using apply()

In [41]:
df["sales_level"] = (
    df["sales"]
    .apply(
        lambda x:
        "High" if x > 2500 else "Low"
    )
)

print(
    df[
        ["sales", "sales_level"]
    ].head()
)

   sales sales_level
0   3863        High
1   1953         Low
2   4759        High
3   1391         Low
4   3681        High


Method Chaining

Method chaining creates readable analysis pipelines

In [42]:
result = (
    df
    .query("sales > 1000")
    .groupby("product")
    .agg(
        avg_sales=("sales", "mean")
    )
    .sort_values(
        "avg_sales",
        ascending=False
    )
)

print(result)

           avg_sales
product             
Tablet   3148.400000
Laptop   3071.555556
Phone    2954.458333


Group Activity 1: Regional Sales Analysis Challenge

Topic: GroupBy, Aggregation, and Filtering

Learning Goal

Use GroupBy operations to discover business insights from sales data.

Group Size

3–4 students

Dataset

Use the a dataset from Kaggle

Task

Each group acts as a data analytics team hired by a company.

In [ ]:
# Questions to answer

# 1. Which region generated the highest total sales?

# 2. Which product generated the highest average sales?

# 3. For each region, determine:
#    - total sales
#    - average sales
#    - maximum sale

# 4. Find all transactions with sales > 4000.

# 5. Present three business recommendations.

Group Activity 2: Sales Dashboard with Pivot Tables

Topic: Pivot Tables and Data Visualization

Learning Goal

Transform raw data into management reports.

Group Size

3–4 students

Scenario

Your team works for a retail company. Management wants a sales summary dashboard.

Tasks
Step 1: Create a Pivot Table

In [43]:
pivot = pd.pivot_table(
    df,
    values="sales",
    index="region",
    columns="product",
    aggfunc="sum"
)

print(pivot)

product  Laptop  Phone  Tablet
region                        
East      30893  18904   33588
North     43554  38484   37576
West      11921  16543   39984


1. Which region sells the most laptops?
2. Which region sells the most phones?
3. Which product is strongest overall?
4. Which region appears weakest?

tep 3: Create a Visualization



Group Activity 3: Time-Series Data Investigation

Team Roles

Divide students into groups of 4:

Student 1: Data Engineer
Load dataset

Check data quality

Handle missing values

Student 2: Data Analyst

Generate descriptive statistics

Explore distributions

Student 3: Time Series Specialist

Analyze trends and seasonality

Compute rolling averages

Student 4: Presenter

Summarize findings

Create visualizations

Present conclusions